In [52]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_regression

In [53]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "cleaned_data.csv"

print("Project root:", PROJECT_ROOT)
print("Input file:", INPUT_PATH)

Project root: /Users/gazifayazwani/Desktop/MLOps
Input file: /Users/gazifayazwani/Desktop/MLOps/data/processed/cleaned_data.csv


In [54]:
import pandas as pd
import numpy as np

from pathlib import Path

In [55]:
df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["datetime"]
)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (34139, 18)


,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,datetime
0,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin,2013-03-01 00:00:00
1,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin,2013-03-01 01:00:00
2,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin,2013-03-01 02:00:00
3,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin,2013-03-01 03:00:00
4,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin,2013-03-01 04:00:00


In [56]:
df = df.sort_values(
    ["station", "datetime"]
).reset_index(drop=True)

# Check duplicate station timestamps
print(
    "Duplicate timestamps:",
    df.duplicated(["station", "datetime"]).sum()
)

Duplicate timestamps: 0


In [57]:
df["day_of_week"] = df["datetime"].dt.dayofweek

df[["datetime", "day_of_week"]].head()

,datetime,day_of_week
0,2013-03-01 00:00:00,4
1,2013-03-01 01:00:00,4
2,2013-03-01 02:00:00,4
3,2013-03-01 03:00:00,4
4,2013-03-01 04:00:00,4


In [58]:
df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

df[["datetime", "day_of_week", "is_weekend"]].head()

,datetime,day_of_week,is_weekend
0,2013-03-01 00:00:00,4,0
1,2013-03-01 01:00:00,4,0
2,2013-03-01 02:00:00,4,0
3,2013-03-01 03:00:00,4,0
4,2013-03-01 04:00:00,4,0


In [59]:
def get_season(month):

    if month in [12, 1, 2]:
        return "Winter"

    elif month in [3, 4, 5]:
        return "Spring"

    elif month in [6, 7, 8]:
        return "Summer"

    else:
        return "Autumn"


df["season"] = df["month"].apply(get_season)

df[["month", "season"]].head(10)

,month,season
0,3,Spring
1,3,Spring
2,3,Spring
3,3,Spring
4,3,Spring
5,3,Spring
6,3,Spring
7,3,Spring
8,3,Spring
9,3,Spring


In [60]:
def get_time_of_day(hour):

    if 5 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 17:
        return "Afternoon"

    elif 17 <= hour < 21:
        return "Evening"

    else:
        return "Night"


df["time_of_day"] = df["hour"].apply(get_time_of_day)

In [61]:
df[
    [
        "datetime",
        "hour",
        "day_of_week",
        "is_weekend",
        "time_of_day",
        "PM2.5"
    ]
].head(10)

,datetime,hour,day_of_week,is_weekend,time_of_day,PM2.5
0,2013-03-01 00:00:00,0,4,0,Night,4.0
1,2013-03-01 01:00:00,1,4,0,Night,8.0
2,2013-03-01 02:00:00,2,4,0,Night,7.0
3,2013-03-01 03:00:00,3,4,0,Night,6.0
4,2013-03-01 04:00:00,4,4,0,Night,3.0
5,2013-03-01 05:00:00,5,4,0,Morning,5.0
6,2013-03-01 06:00:00,6,4,0,Morning,3.0
7,2013-03-01 07:00:00,7,4,0,Morning,3.0
8,2013-03-01 08:00:00,8,4,0,Morning,3.0
9,2013-03-01 09:00:00,9,4,0,Morning,3.0


In [62]:
df = df.sort_values(
    ["station", "datetime"]
).reset_index(drop=True)

df["PM2.5_rolling_3"] = (
    df.groupby("station")["PM2.5"]
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

df["PM2.5_rolling_24"] = (
    df.groupby("station")["PM2.5"]
    .transform(lambda x: x.shift(1).rolling(24).mean())
)

In [63]:
df = df.dropna(
    subset=["PM2.5_rolling_3", "PM2.5_rolling_24"]
).reset_index(drop=True)

print(df.shape)

(34115, 24)


In [64]:
numeric_features = [
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "WSPM",
    "PM2.5_rolling_3",
    "PM2.5_rolling_24"
]

categorical_features = [
    "wd",
    "station",
    "season"
]

X = df[numeric_features + categorical_features]

y = df["PM2.5"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (34115, 15)
y shape: (34115,)


In [65]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    shuffle=False
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (27292, 15)
Testing data: (6823, 15)


In [66]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [67]:
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", SelectKBest(
        score_func=f_regression,
        k=10
    ))
])

In [68]:
X_train_processed = model_pipeline.fit_transform(X_train, y_train)

X_test_processed = model_pipeline.transform(X_test)

print("Selected training features:", X_train_processed.shape)
print("Selected testing features:", X_test_processed.shape)

Selected training features: (27292, 10)
Selected testing features: (6823, 10)


In [69]:
feature_names = (
    model_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

selector = model_pipeline.named_steps["feature_selection"]

selected_features = feature_names[selector.get_support()]

print("Selected Features:")

for feature in selected_features:
    print(feature)

Selected Features:
num__PM10
num__SO2
num__NO2
num__CO
num__O3
num__DEWP
num__WSPM
num__PM2.5_rolling_3
num__PM2.5_rolling_24
cat__wd_NW


In [70]:
X_train_final = pd.DataFrame(
    X_train_processed,
    columns=selected_features
)

X_test_final = pd.DataFrame(
    X_test_processed,
    columns=selected_features
)

X_train_final.to_csv(
    FEATURES_DIR / "X_train.csv",
    index=False
)

X_test_final.to_csv(
    FEATURES_DIR / "X_test.csv",
    index=False
)

y_train.to_frame().to_csv(
    FEATURES_DIR / "y_train.csv",
    index=False
)

y_test.to_frame().to_csv(
    FEATURES_DIR / "y_test.csv",
    index=False
)

# Save fitted preprocessing and feature-selection pipeline
joblib.dump(
    model_pipeline,
    FEATURES_DIR / "preprocessing_pipeline.pkl"
)

print("All datasets saved successfully!")

All datasets saved successfully!
